# EdgeVerify — Full Reproduction Notebook (v5)

Reproduces every experiment in the thesis, including **multi-seed** placement and federated studies.

**Runtime → Run all.** A **GPU runtime** is recommended (multi-seed parts are many training runs).


## 0. Install


In [ ]:
!pip install -q torch torchvision scikit-learn matplotlib

## 1. Recreate the repo files


In [ ]:
%%writefile model.py
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

def compute_stable_local_loss(projected_student, teacher_target, layer_output,
                              variance_threshold=1.0, alpha=1.0, beta=0.01,
                              shared_cov=False, var_on="context", cov_on=None,
                              return_components=False):
    """
    Computes regularized local distillation loss using VicReg-style constraints
    to prevent dimensional collapse.

    The default coefficients (alpha=1.0, beta=0.01) reproduce the original
    formulation exactly. They are exposed as arguments so that the
    unregularized baseline (alpha=beta=0) required for the collapse ablation
    can share this single code path. Set return_components=True to obtain the
    per-term breakdown for logging.

    Placement of the two regularizers is configurable, to support the
    regularizer-placement study. ``var_on`` and ``cov_on`` each select the
    tensor a penalty acts on: ``"context"`` = the context embedding
    ``layer_output`` (s_t), ``"pred"`` = the predictor output
    ``projected_student`` (s_pred).

    Backward compatibility: ``var_on`` defaults to ``"context"``. If ``cov_on``
    is left as ``None`` it is resolved from ``shared_cov`` -- ``"context"`` when
    ``shared_cov=True`` (the shared-embedding variant), otherwise ``"pred"``
    (the original wiring). An explicit ``cov_on`` overrides ``shared_cov``.
    """
    tensors = {"context": layer_output, "pred": projected_student}
    if cov_on is None:
        cov_on = "context" if shared_cov else "pred"
    var_tensor = tensors[var_on]
    cov_tensor = tensors[cov_on]

    # 1. Base Distillation Loss (MSE against target)
    distill_loss = F.mse_loss(projected_student, teacher_target)

    # 2. Variance Constraint (Hinge loss on batch standard deviation)
    std_student = torch.sqrt(var_tensor.var(dim=0) + 1e-4)
    variance_loss = torch.mean(F.relu(variance_threshold - std_student))

    # 3. Covariance Regularization (Feature decorrelation)
    centered_student = cov_tensor - cov_tensor.mean(dim=0)
    batch_size = cov_tensor.size(0)
    cov_matrix = (centered_student.T @ centered_student) / (batch_size - 1)
    diag_mask = torch.eye(cov_matrix.size(0), device=cov_tensor.device)
    covariance_loss = (cov_matrix * (1 - diag_mask)).pow(2).sum() / cov_matrix.size(0)

    total = distill_loss + alpha * variance_loss + beta * covariance_loss
    if return_components:
        return total, {
            "distill": distill_loss.item(),
            "var": variance_loss.item(),
            "cov": covariance_loss.item(),
            "total": total.item(),
        }
    return total

class VisionJEPA(nn.Module):
    def __init__(self, img_channels=3, latent_dim=256, ema_decay=0.999):
        super().__init__()
        self.latent_dim = latent_dim
        self.ema_decay = ema_decay

        # Context Encoder (f_theta)
        self.context_encoder = nn.Sequential(
            nn.Conv2d(img_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, latent_dim, kernel_size=3, stride=2, padding=1),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

        # Target Encoder (f_theta_bar) - Updated via EMA
        self.target_encoder = copy.deepcopy(self.context_encoder)
        for p in self.target_encoder.parameters():
            p.requires_grad = False

        # Action/Bounding-Box Encoder
        self.action_encoder = nn.Sequential(
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim)
        )

        # Latent Predictor Block (p_psi)
        self.predictor = nn.Sequential(
            nn.Linear(latent_dim * 2, 512),
            nn.ReLU(),
            nn.Linear(512, latent_dim)
        )

    @torch.no_grad()
    def update_target_encoder(self):
        for p_ctx, p_tgt in zip(self.context_encoder.parameters(), self.target_encoder.parameters()):
            p_tgt.data.mul_(self.ema_decay).add_(p_ctx.data, alpha=1.0 - self.ema_decay)

    def forward(self, partial_images, full_images, spatial_actions):
        s_t = self.context_encoder(partial_images)
        with torch.no_grad():
            s_target = self.target_encoder(full_images)
        a_t = self.action_encoder(spatial_actions)

        combined_latent = torch.cat([s_t, a_t], dim=-1)
        s_predicted = self.predictor(combined_latent)

        return s_predicted, s_target, s_t


In [ ]:
%%writefile dataset.py
"""
Dataset generators and masking loaders for EdgeVerify.

Two sources are provided:

  * ``StructuredJEPADataset`` -- procedurally generated 64x64 RGB images with
    genuine spatial structure (background gradients plus randomly placed
    rectangles and circles). Unlike i.i.d. uniform noise, these images contain
    predictable local structure, so a context encoder can learn a non-trivial
    representation and the collapse ablation is meaningful.

  * ``DigitsJEPADataset`` -- the scikit-learn handwritten-digits dataset
    (real 8x8 images, 1797 samples), upsampled to 64x64 and replicated to
    three channels. This is real (non-synthetic) data available offline, used
    as a sanity check.

Each sample is a triple ``(partial_image, full_image, spatial_action)`` matching
the signature of ``VisionJEPA.forward``. The partial (context) view is produced
by zeroing a rectangular region of the full image; the spatial action is the
normalized bounding box ``[x, y, w, h]`` of that masked region, so the predictor
is conditioned on the location it must reconstruct in latent space.
"""

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset


def _apply_mask(img, rng, min_frac=0.35, max_frac=0.6):
    """Zero a random rectangular region; return (partial, action[x,y,w,h] normalized)."""
    _, H, W = img.shape
    w = int(rng.uniform(min_frac, max_frac) * W)
    h = int(rng.uniform(min_frac, max_frac) * H)
    x = int(rng.integers(0, W - w + 1))
    y = int(rng.integers(0, H - h + 1))
    partial = img.clone()
    partial[:, y:y + h, x:x + w] = 0.0
    action = torch.tensor([x / W, y / H, w / W, h / H], dtype=torch.float32)
    return partial, action


def _make_structured_image(rng, size=64):
    """Background gradient + a few random rectangles and circles, values in [0,1]."""
    img = np.zeros((3, size, size), dtype=np.float32)
    # Background gradient between two random colours.
    c0 = rng.uniform(0, 1, size=3)
    c1 = rng.uniform(0, 1, size=3)
    if rng.random() < 0.5:
        ramp = np.linspace(0, 1, size)[None, :]        # horizontal
    else:
        ramp = np.linspace(0, 1, size)[:, None]        # vertical
    for c in range(3):
        img[c] = c0[c] + (c1[c] - c0[c]) * ramp
    # Random rectangles.
    yy, xx = np.mgrid[0:size, 0:size]
    for _ in range(int(rng.integers(1, 4))):
        rw, rh = rng.integers(8, 28), rng.integers(8, 28)
        rx, ry = rng.integers(0, size - rw), rng.integers(0, size - rh)
        col = rng.uniform(0, 1, size=3)
        img[:, ry:ry + rh, rx:rx + rw] = col[:, None, None]
    # Random filled circles.
    for _ in range(int(rng.integers(1, 4))):
        cx, cy = rng.integers(10, size - 10), rng.integers(10, size - 10)
        r = rng.integers(5, 14)
        col = rng.uniform(0, 1, size=3)
        disk = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
        for c in range(3):
            img[c][disk] = col[c]
    return torch.from_numpy(np.clip(img, 0.0, 1.0))


class StructuredJEPADataset(Dataset):
    def __init__(self, n_samples=1024, size=64, seed=0):
        rng = np.random.default_rng(seed)
        self.samples = []
        for _ in range(n_samples):
            full = _make_structured_image(rng, size)
            partial, action = _apply_mask(full, rng)
            self.samples.append((partial, full, action))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


class DigitsJEPADataset(Dataset):
    def __init__(self, size=64, seed=0):
        from sklearn.datasets import load_digits
        rng = np.random.default_rng(seed)
        digits = load_digits()
        imgs = digits.images.astype(np.float32) / 16.0          # (1797, 8, 8), scaled to [0,1]
        t = torch.from_numpy(imgs).unsqueeze(1)                   # (N, 1, 8, 8)
        t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
        t = t.repeat(1, 3, 1, 1)                                  # (N, 3, size, size)
        self.samples = []
        for i in range(t.size(0)):
            full = t[i].contiguous()
            partial, action = _apply_mask(full, rng)
            self.samples.append((partial, full, action))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


class CIFARJEPADataset(Dataset):
    """CIFAR-10 subset (requires network access to download via torchvision)."""
    def __init__(self, n_samples=2000, size=64, seed=0, root="./cifar"):
        import torchvision
        rng = np.random.default_rng(seed)
        tv = torchvision.datasets.CIFAR10(root=root, train=True, download=True)
        data = torch.from_numpy(tv.data[:n_samples]).float().permute(0, 3, 1, 2) / 255.0
        data = F.interpolate(data, size=(size, size), mode="bilinear", align_corners=False)
        self.samples = []
        for i in range(data.size(0)):
            full = data[i].contiguous()
            partial, action = _apply_mask(full, rng)
            self.samples.append((partial, full, action))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def get_dataset(name, **kwargs):
    name = name.lower()
    if name in ("structured", "shapes"):
        return StructuredJEPADataset(**kwargs)
    if name in ("digits", "sklearn"):
        return DigitsJEPADataset(**kwargs)
    if name in ("cifar", "cifar10", "cifar-10"):
        return CIFARJEPADataset(**kwargs)
    raise ValueError(f"Unknown dataset: {name!r}")


In [ ]:
%%writefile evaluate.py
"""
Evaluation protocol for EdgeVerify.

Runs the collapse ablation (full VICReg-style objective vs. an unregularized
alpha=beta=0 baseline) on each available dataset, and measures:

  RQ1 (collapse) : mean per-dimension embedding std (sigma_bar) and the
                   effective rank (participation ratio) of the context-embedding
                   covariance. Higher is healthier; collapse drives both to ~0/1.
  RQ2 (efficiency): parameter count, parameter memory, single-sample inference
                   latency (mean +/- std), and peak process RSS during training.
  RQ3 (convergence): per-epoch loss components.

Outputs a results JSON and PNG figures for inclusion in the thesis.
"""

import json
import os
import time

import numpy as np
import torch
from torch.utils.data import DataLoader

from model import VisionJEPA, compute_stable_local_loss
from dataset import get_dataset


def _rss_mb():
    """Resident set size of this process in MB, read from /proc/self/status."""
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return float(line.split()[1]) / 1024.0  # kB -> MB
    except FileNotFoundError:
        pass
    return float("nan")


def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)


def train_model(dataset, epochs, alpha, beta, batch_size=64, lr=1e-3, wd=1e-4, seed=0,
                shared_cov=False, var_on="context", cov_on=None):
    set_seed(seed)
    device = torch.device("cpu")
    model = VisionJEPA().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    history = []
    peak_rss = _rss_mb()
    for epoch in range(epochs):
        agg = {"distill": 0.0, "var": 0.0, "cov": 0.0, "total": 0.0}
        n = 0
        for partial, full, action in loader:
            partial, full, action = partial.to(device), full.to(device), action.to(device)
            optimizer.zero_grad()
            s_pred, s_tgt, s_t = model(partial, full, action)
            loss, comps = compute_stable_local_loss(
                s_pred, s_tgt.detach(), s_t,
                variance_threshold=1.0, alpha=alpha, beta=beta,
                shared_cov=shared_cov, var_on=var_on, cov_on=cov_on,
                return_components=True)
            loss.backward()
            optimizer.step()
            model.update_target_encoder()
            for k in agg:
                agg[k] += comps[k]
            n += 1
            peak_rss = max(peak_rss, _rss_mb())
        history.append({k: agg[k] / max(n, 1) for k in agg})
    return model, history, peak_rss


@torch.no_grad()
def collapse_metrics(model, dataset, max_samples=512):
    model.eval()
    device = torch.device("cpu")
    loader = DataLoader(dataset, batch_size=128, shuffle=False)
    embs = []
    seen = 0
    for partial, full, action in loader:
        s_t = model.context_encoder(partial.to(device))
        embs.append(s_t)
        seen += s_t.size(0)
        if seen >= max_samples:
            break
    E = torch.cat(embs, dim=0)[:max_samples]                 # (M, d)
    sigma_bar = torch.sqrt(E.var(dim=0) + 1e-8).mean().item()
    Ec = E - E.mean(dim=0, keepdim=True)
    cov = (Ec.T @ Ec) / (E.size(0) - 1)
    eig = torch.linalg.eigvalsh(cov).clamp(min=0)
    s1, s2 = eig.sum().item(), (eig ** 2).sum().item()
    eff_rank = (s1 ** 2) / s2 if s2 > 0 else 0.0             # participation ratio
    return {"sigma_bar": sigma_bar, "effective_rank": eff_rank, "latent_dim": E.size(1)}


@torch.no_grad()
def efficiency_metrics(model, n_runs=50, warmup=10):
    model.eval()
    device = torch.device("cpu")
    n_params = sum(p.numel() for p in model.parameters())
    param_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2)
    partial = torch.rand(1, 3, 64, 64)
    full = torch.rand(1, 3, 64, 64)
    action = torch.rand(1, 4)
    for _ in range(warmup):
        model(partial, full, action)
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        model(partial, full, action)
        times.append((time.perf_counter() - t0) * 1000.0)     # ms
    times = np.array(times)
    return {
        "params": int(n_params),
        "param_memory_mb": round(param_mb, 3),
        "latency_ms_mean": round(float(times.mean()), 3),
        "latency_ms_std": round(float(times.std()), 3),
    }


def run_dataset(name, epochs, ds_kwargs, out_img_dir):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    dataset = get_dataset(name, **ds_kwargs)
    results = {}
    curves = {}
    for tag, (alpha, beta) in {"full": (1.0, 0.01), "unregularized": (0.0, 0.0)}.items():
        model, history, peak_rss = train_model(dataset, epochs, alpha, beta, seed=0)
        col = collapse_metrics(model, dataset)
        eff = efficiency_metrics(model)
        results[tag] = {
            "alpha": alpha, "beta": beta,
            "final_loss": history[-1],
            "collapse": col,
            "efficiency": eff,
            "peak_rss_mb": round(peak_rss, 1),
        }
        curves[tag] = history

    # Loss-curve figure (total loss per epoch, both configs).
    plt.figure(figsize=(6, 4))
    for tag in curves:
        plt.plot([h["total"] for h in curves[tag]], label=tag, linewidth=2)
    plt.xlabel("Epoch"); plt.ylabel(r"$\mathcal{L}_{total}$")
    plt.title(f"Convergence on {name} data"); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    loss_path = os.path.join(out_img_dir, f"loss_curves_{name}.png")
    plt.savefig(loss_path, dpi=150); plt.close()

    return results, loss_path


def main():
    here = os.path.dirname(os.path.abspath(__file__))
    img_dir = os.path.abspath(os.path.join(here, "..", "latex_report", "images"))
    res_dir = os.path.abspath(os.path.join(here, "..", "results"))
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(res_dir, exist_ok=True)

    all_results = {}
    plans = [
        ("structured", 40, {"n_samples": 768, "seed": 0}),
        ("digits", 25, {"seed": 0}),
    ]
    for name, epochs, kw in plans:
        print(f"\n=== Running {name} (epochs={epochs}) ===", flush=True)
        res, _ = run_dataset(name, epochs, kw, img_dir)
        all_results[name] = res
        for tag, r in res.items():
            c, e = r["collapse"], r["efficiency"]
            print(f"  [{tag:13s}] total={r['final_loss']['total']:.4f} "
                  f"sigma_bar={c['sigma_bar']:.4f} eff_rank={c['effective_rank']:.2f}/"
                  f"{c['latent_dim']} lat={e['latency_ms_mean']:.2f}ms "
                  f"params={e['params']:,} peakRSS={r['peak_rss_mb']:.0f}MB", flush=True)

    # Collapse comparison bar chart (structured dataset).
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    for name in all_results:
        r = all_results[name]
        fig, ax = plt.subplots(1, 2, figsize=(8, 3.4))
        tags = ["unregularized", "full"]
        ax[0].bar(tags, [r[t]["collapse"]["sigma_bar"] for t in tags],
                  color=["#c0504d", "#4f81bd"])
        ax[0].set_title(r"Mean embedding std $\bar{\sigma}$")
        ax[1].bar(tags, [r[t]["collapse"]["effective_rank"] for t in tags],
                  color=["#c0504d", "#4f81bd"])
        ax[1].set_title("Effective rank")
        for a in ax:
            a.grid(alpha=0.3, axis="y")
        fig.suptitle(f"Collapse indicators ({name} data)")
        fig.tight_layout()
        fig.savefig(os.path.join(img_dir, f"collapse_bars_{name}.png"), dpi=150)
        plt.close(fig)

    with open(os.path.join(res_dir, "results.json"), "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved results.json and figures to {res_dir} and {img_dir}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate_multiseed.py
"""
Multi-seed robustness evaluation (addresses the single-seed threat to validity).

For each dataset and each configuration (full objective vs. unregularized),
the model is trained under several random training seeds. Data content is held
fixed (data seed = 0); the varied seed controls weight initialization and
minibatch ordering, isolating training stochasticity. We report the mean and
standard deviation of the collapse indicators across seeds.
"""

import json
import os

import numpy as np
import torch

from model import VisionJEPA  # noqa: F401  (ensures import path is valid)
from dataset import get_dataset
from evaluate import train_model, collapse_metrics

SEEDS = [0, 1, 2, 3, 4]
PLANS = [
    ("structured", 40, {"n_samples": 768, "seed": 0}),
    ("digits", 25, {"seed": 0}),
]
CONFIGS = {"full": (1.0, 0.01), "unregularized": (0.0, 0.0)}


def agg(values):
    a = np.array(values, dtype=float)
    return {"mean": float(a.mean()), "std": float(a.std()), "n": len(a), "values": [round(v, 4) for v in values]}


def main():
    here = os.path.dirname(os.path.abspath(__file__))
    res_dir = os.path.abspath(os.path.join(here, "..", "results"))
    img_dir = os.path.abspath(os.path.join(here, "..", "latex_report", "images"))
    os.makedirs(res_dir, exist_ok=True)
    os.makedirs(img_dir, exist_ok=True)

    results = {}
    for name, epochs, ds_kwargs in PLANS:
        dataset = get_dataset(name, **ds_kwargs)
        results[name] = {}
        for tag, (alpha, beta) in CONFIGS.items():
            sig, rank, tot = [], [], []
            for seed in SEEDS:
                model, history, _ = train_model(dataset, epochs, alpha, beta, seed=seed)
                col = collapse_metrics(model, dataset)
                sig.append(col["sigma_bar"])
                rank.append(col["effective_rank"])
                tot.append(history[-1]["total"])
                print(f"[{name:10s}/{tag:13s}] seed={seed} "
                      f"sigma_bar={col['sigma_bar']:.4f} eff_rank={col['effective_rank']:.3f} "
                      f"total={history[-1]['total']:.4f}", flush=True)
            results[name][tag] = {
                "sigma_bar": agg(sig),
                "effective_rank": agg(rank),
                "total_loss": agg(tot),
            }

    with open(os.path.join(res_dir, "multiseed.json"), "w") as f:
        json.dump({"seeds": SEEDS, "results": results}, f, indent=2)

    # Figure: mean +/- std bars for effective rank and sigma_bar.
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    for name in results:
        fig, ax = plt.subplots(1, 2, figsize=(8, 3.4))
        tags = ["unregularized", "full"]
        colors = ["#c0504d", "#4f81bd"]
        for j, metric, title in [(0, "sigma_bar", r"Mean embedding std $\bar{\sigma}$"),
                                 (1, "effective_rank", "Effective rank")]:
            means = [results[name][t][metric]["mean"] for t in tags]
            stds = [results[name][t][metric]["std"] for t in tags]
            ax[j].bar(tags, means, yerr=stds, capsize=6, color=colors)
            ax[j].set_title(title)
            ax[j].grid(alpha=0.3, axis="y")
        fig.suptitle(f"Robustness over {len(SEEDS)} seeds ({name} data): mean $\\pm$ std")
        fig.tight_layout()
        fig.savefig(os.path.join(img_dir, f"multiseed_{name}.png"), dpi=150)
        plt.close(fig)

    print("\nSUMMARY (mean +/- std over seeds):", flush=True)
    for name in results:
        for tag in CONFIGS:
            r = results[name][tag]
            print(f"  {name:10s} {tag:13s}  "
                  f"sigma_bar={r['sigma_bar']['mean']:.3f}+/-{r['sigma_bar']['std']:.3f}  "
                  f"eff_rank={r['effective_rank']['mean']:.2f}+/-{r['effective_rank']['std']:.2f}", flush=True)
    print("DONE", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate_fixes.py
"""
Evaluates the two experimental fixes:

  1. Shared-embedding variant: apply the covariance penalty to the context
     embedding s_t (same tensor as the variance penalty) instead of to the
     prediction s_pred, testing whether it restores effective rank.
  2. Linear probe: freeze the trained context encoder and fit a linear
     classifier on labels (digits), measuring whether the representation is
     actually useful for a downstream task -- a ground-truth signal the
     intrinsic collapse metrics lack.

Three configurations are compared on structured and digits data:
  - unregularized (alpha=beta=0)
  - full          (original wiring: variance on s_t, covariance on s_pred)
  - shared        (variance and covariance both on s_t)
"""

import json
import os

import numpy as np
import torch
import torch.nn.functional as F

from dataset import get_dataset
from evaluate import train_model, collapse_metrics

CONFIGS = {
    "unregularized": dict(alpha=0.0, beta=0.0, shared_cov=False),
    "full":          dict(alpha=1.0, beta=0.01, shared_cov=False),
    "shared":        dict(alpha=1.0, beta=1.0, shared_cov=True),
}
PLANS = [
    ("structured", 40, {"n_samples": 768, "seed": 0}),
    ("digits", 25, {"seed": 0}),
]


def digits_features_and_labels(size=64):
    from sklearn.datasets import load_digits
    d = load_digits()
    imgs = torch.from_numpy(d.images.astype("float32") / 16.0).unsqueeze(1)
    imgs = F.interpolate(imgs, size=(size, size), mode="bilinear", align_corners=False)
    imgs = imgs.repeat(1, 3, 1, 1)
    return imgs, d.target


@torch.no_grad()
def linear_probe(model, imgs, labels, seed=0):
    """Freeze encoder, extract context features on full images, fit a linear classifier."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    model.eval()
    feats = model.context_encoder(imgs).numpy()
    Xtr, Xte, ytr, yte = train_test_split(feats, labels, test_size=0.3,
                                          random_state=seed, stratify=labels)
    scaler = StandardScaler().fit(Xtr)
    clf = LogisticRegression(max_iter=3000)
    clf.fit(scaler.transform(Xtr), ytr)
    return float(clf.score(scaler.transform(Xte), yte))


def main():
    here = os.path.dirname(os.path.abspath(__file__))
    res_dir = os.path.abspath(os.path.join(here, "..", "results"))
    os.makedirs(res_dir, exist_ok=True)

    digit_imgs, digit_labels = digits_features_and_labels()
    results = {}
    for name, epochs, ds_kwargs in PLANS:
        dataset = get_dataset(name, **ds_kwargs)
        results[name] = {}
        for tag, cfg in CONFIGS.items():
            model, history, _ = train_model(dataset, epochs, cfg["alpha"], cfg["beta"],
                                            seed=0, shared_cov=cfg["shared_cov"])
            col = collapse_metrics(model, dataset)
            probe = linear_probe(model, digit_imgs, digit_labels) if name == "digits" else None
            results[name][tag] = {
                "config": cfg,
                "sigma_bar": round(col["sigma_bar"], 4),
                "effective_rank": round(col["effective_rank"], 3),
                "probe_accuracy": (round(probe, 4) if probe is not None else None),
            }
            msg = (f"[{name:10s}/{tag:13s}] sigma_bar={col['sigma_bar']:.3f} "
                   f"eff_rank={col['effective_rank']:.2f}/256")
            if probe is not None:
                msg += f"  probe_acc={probe:.3f}"
            print(msg, flush=True)

    with open(os.path.join(res_dir, "fixes.json"), "w") as f:
        json.dump(results, f, indent=2)
    print("DONE", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate_placement.py
"""
Regularizer-placement study.

The variance and covariance penalties can each be applied to the context
embedding s_t ("context") or the predictor output s_pred ("pred"). This script
sweeps the full 2x2 grid of placements at fixed weights (alpha=beta=1.0) and
reports, for the resulting frozen context encoder: mean embedding std, effective
rank, the final variance-loss term (to show whether the variance criterion is
"satisfied"), and -- on digits -- linear-probe accuracy.

The aim is to determine which placement is responsible for collapse resistance,
turning the earlier "shared-embedding" observation into a systematic result.
"""

import json
import os

import torch

from dataset import get_dataset
from evaluate import train_model, collapse_metrics
from evaluate_fixes import digits_features_and_labels, linear_probe

GRID = [
    ("var s_t, cov s_pred (original)", "context", "pred"),
    ("var s_t, cov s_t (shared)",      "context", "context"),
    ("var s_pred, cov s_pred",         "pred",    "pred"),
    ("var s_pred, cov s_t (swapped)",  "pred",    "context"),
]
PLANS = [
    ("structured", 40, {"n_samples": 768, "seed": 0}),
    ("digits", 25, {"seed": 0}),
]
ALPHA = BETA = 1.0


def main():
    here = os.path.dirname(os.path.abspath(__file__))
    res_dir = os.path.abspath(os.path.join(here, "..", "results"))
    os.makedirs(res_dir, exist_ok=True)
    digit_imgs, digit_labels = digits_features_and_labels()

    results = {}
    for name, epochs, kw in PLANS:
        dataset = get_dataset(name, **kw)
        results[name] = {}
        for label, von, con in GRID:
            model, history, _ = train_model(dataset, epochs, ALPHA, BETA, seed=0,
                                            var_on=von, cov_on=con)
            col = collapse_metrics(model, dataset)
            probe = linear_probe(model, digit_imgs, digit_labels) if name == "digits" else None
            results[name][label] = {
                "var_on": von, "cov_on": con,
                "sigma_bar": round(col["sigma_bar"], 4),
                "effective_rank": round(col["effective_rank"], 3),
                "final_var_loss": round(history[-1]["var"], 4),
                "probe_accuracy": (round(probe, 4) if probe is not None else None),
            }
            msg = (f"[{name:10s}] {label:32s} sigma_bar={col['sigma_bar']:.3f} "
                   f"eff_rank={col['effective_rank']:6.2f}/256 var_loss={history[-1]['var']:.3f}")
            if probe is not None:
                msg += f" probe_acc={probe:.3f}"
            print(msg, flush=True)

    with open(os.path.join(res_dir, "placement.json"), "w") as f:
        json.dump({"alpha": ALPHA, "beta": BETA, "results": results}, f, indent=2)
    print("DONE", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate_placement_ms.py
"""Multi-seed regularizer-placement study (5 seeds). Aggregates effective rank
and (digits) linear-probe accuracy as mean +/- std across seeds."""
import json, os
import numpy as np
from dataset import get_dataset
from evaluate import train_model, collapse_metrics
from evaluate_fixes import digits_features_and_labels, linear_probe

GRID = [
    ("var s_t, cov s_pred (original)", "context", "pred"),
    ("var s_t, cov s_t (shared)",      "context", "context"),
    ("var s_pred, cov s_pred",         "pred",    "pred"),
    ("var s_pred, cov s_t (swapped)",  "pred",    "context"),
]
PLANS = [("structured", 40, {"n_samples": 768, "seed": 0}), ("digits", 25, {"seed": 0})]
SEEDS = [0, 1, 2, 3, 4]
A = B = 1.0


def agg(v):
    a = np.array(v, float)
    return {"mean": round(float(a.mean()), 3), "std": round(float(a.std()), 3),
            "values": [round(x, 3) for x in v]}


def main():
    here = os.path.dirname(os.path.abspath(__file__))
    res_dir = os.path.abspath(os.path.join(here, "..", "results"))
    di, dl = digits_features_and_labels()
    results = {}
    for name, ep, kw in PLANS:
        ds = get_dataset(name, **kw)
        results[name] = {}
        for label, von, con in GRID:
            ranks, accs = [], []
            for s in SEEDS:
                m, h, _ = train_model(ds, ep, A, B, seed=s, var_on=von, cov_on=con)
                ranks.append(collapse_metrics(m, ds)["effective_rank"])
                if name == "digits":
                    accs.append(linear_probe(m, di, dl))
            results[name][label] = {"effective_rank": agg(ranks),
                                    "probe_accuracy": (agg(accs) if name == "digits" else None)}
            r = results[name][label]
            msg = f"[{name:10s}] {label:32s} eff_rank={r['effective_rank']['mean']:6.2f}+/-{r['effective_rank']['std']:.2f}"
            if accs:
                msg += f" probe={r['probe_accuracy']['mean']:.3f}+/-{r['probe_accuracy']['std']:.3f}"
            print(msg, flush=True)
    with open(os.path.join(res_dir, "placement_ms.json"), "w") as f:
        json.dump({"seeds": SEEDS, "alpha": A, "beta": B, "results": results}, f, indent=2)
    print("DONE", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate_federated.py
"""
Decentralized (multi-agent) training via FedAvg.

Two nodes hold disjoint shards of the data and train locally; after each
communication round their online parameters (context encoder, action encoder,
predictor -- including BatchNorm statistics) are averaged into a global model
and broadcast back. Target encoders are re-synchronised to the global context
encoder at the start of each round and updated by EMA locally.

The question: does the collapse behaviour observed in the centralized setting
survive decentralized averaging? We compare, for the frozen global context
encoder, effective rank and (on digits) linear-probe accuracy across:
  - centralized, shared-embedding objective (reference)
  - federated (2 nodes), shared-embedding objective
  - federated (2 nodes), original wiring
"""

import copy
import json
import os

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

from model import VisionJEPA, compute_stable_local_loss
from dataset import get_dataset
from evaluate import collapse_metrics, set_seed, train_model
from evaluate_fixes import digits_features_and_labels, linear_probe

ONLINE = ["context_encoder", "action_encoder", "predictor"]


def _get_online(model):
    return {n: copy.deepcopy(getattr(model, n).state_dict()) for n in ONLINE}


def _set_online(model, states):
    for n in ONLINE:
        getattr(model, n).load_state_dict(states[n])


def _average(states_list):
    avg = {}
    for n in ONLINE:
        avg[n] = {}
        for k in states_list[0][n].keys():
            avg[n][k] = sum(s[n][k].float() for s in states_list) / len(states_list)
    return avg


def _local_train(model, subset, epochs, alpha, beta, cfg, batch_size=64, lr=1e-3, wd=1e-4):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loader = DataLoader(subset, batch_size=batch_size, shuffle=True, drop_last=True)
    for _ in range(epochs):
        for partial, full, action in loader:
            opt.zero_grad()
            s_pred, s_tgt, s_t = model(partial, full, action)
            loss = compute_stable_local_loss(s_pred, s_tgt.detach(), s_t,
                                             alpha=alpha, beta=beta,
                                             var_on=cfg["var_on"], cov_on=cfg["cov_on"])
            loss.backward()
            opt.step()
            model.update_target_encoder()
    return model


def federated_train(dataset, n_nodes, rounds, local_epochs, alpha, beta, cfg, seed=0):
    set_seed(seed)
    idx = np.arange(len(dataset))
    np.random.shuffle(idx)
    shards = [Subset(dataset, idx[i::n_nodes].tolist()) for i in range(n_nodes)]
    global_model = VisionJEPA()
    global_states = _get_online(global_model)
    for _ in range(rounds):
        node_states = []
        for shard in shards:
            node = VisionJEPA()
            _set_online(node, global_states)
            node.target_encoder = copy.deepcopy(node.context_encoder)
            for p in node.target_encoder.parameters():
                p.requires_grad = False
            _local_train(node, shard, local_epochs, alpha, beta, cfg)
            node_states.append(_get_online(node))
        global_states = _average(node_states)
    _set_online(global_model, global_states)
    return global_model


def main():
    here = os.path.dirname(os.path.abspath(__file__))
    res_dir = os.path.abspath(os.path.join(here, "..", "results"))
    os.makedirs(res_dir, exist_ok=True)
    digit_imgs, digit_labels = digits_features_and_labels()

    shared = {"var_on": "context", "cov_on": "context"}
    original = {"var_on": "context", "cov_on": "pred"}

    # (name, epochs_centralized, rounds, local_epochs, ds_kwargs)
    plans = [
        ("structured", 36, 6, 6, {"n_samples": 768, "seed": 0}),
        ("digits", 24, 6, 4, {"seed": 0}),
    ]
    results = {}
    for name, cen_ep, rounds, local_ep, kw in plans:
        ds = get_dataset(name, **kw)
        results[name] = {}

        def record(tag, model):
            col = collapse_metrics(model, ds)
            probe = linear_probe(model, digit_imgs, digit_labels) if name == "digits" else None
            results[name][tag] = {"effective_rank": round(col["effective_rank"], 3),
                                  "sigma_bar": round(col["sigma_bar"], 4),
                                  "probe_accuracy": (round(probe, 4) if probe is not None else None)}
            msg = f"[{name:10s}/{tag:22s}] eff_rank={col['effective_rank']:6.2f}/256 sigma_bar={col['sigma_bar']:.3f}"
            if probe is not None:
                msg += f" probe_acc={probe:.3f}"
            print(msg, flush=True)

        m_cen, _, _ = train_model(ds, cen_ep, 1.0, 1.0, seed=0,
                                  var_on="context", cov_on="context")
        record("centralized-shared", m_cen)
        record("federated-shared", federated_train(ds, 2, rounds, local_ep, 1.0, 1.0, shared, seed=0))
        record("federated-original", federated_train(ds, 2, rounds, local_ep, 1.0, 1.0, original, seed=0))

    with open(os.path.join(res_dir, "federated.json"), "w") as f:
        json.dump(results, f, indent=2)
    print("DONE", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate_federated_ms.py
"""Multi-seed (3 seeds) version of the decentralized study. Aggregates
effective rank and (digits) probe accuracy as mean +/- std across seeds for
centralized-shared, federated-shared, and federated-original."""
import json, os
import numpy as np

from dataset import get_dataset
from evaluate import train_model, collapse_metrics
from evaluate_fixes import digits_features_and_labels, linear_probe
from evaluate_federated import federated_train

SEEDS = [0, 1, 2]
SHARED = {"var_on": "context", "cov_on": "context"}
ORIGINAL = {"var_on": "context", "cov_on": "pred"}
# (name, centralized_epochs, rounds, local_epochs, ds_kwargs)
PLANS = [
    ("structured", 36, 6, 6, {"n_samples": 768, "seed": 0}),
    ("digits", 24, 6, 4, {"seed": 0}),
]


def agg(v):
    a = np.array(v, float)
    return {"mean": round(float(a.mean()), 3), "std": round(float(a.std()), 3),
            "values": [round(x, 3) for x in v]}


def main():
    here = os.path.dirname(os.path.abspath(__file__))
    res_dir = os.path.abspath(os.path.join(here, "..", "results"))
    di, dl = digits_features_and_labels()
    results = {}
    for name, cen_ep, rounds, local_ep, kw in PLANS:
        ds = get_dataset(name, **kw)
        results[name] = {}

        def run(tag, builder):
            ranks, accs = [], []
            for s in SEEDS:
                model = builder(s)
                ranks.append(collapse_metrics(model, ds)["effective_rank"])
                if name == "digits":
                    accs.append(linear_probe(model, di, dl))
            results[name][tag] = {"effective_rank": agg(ranks),
                                  "probe_accuracy": (agg(accs) if name == "digits" else None)}
            r = results[name][tag]
            msg = f"[{name:10s}/{tag:20s}] eff_rank={r['effective_rank']['mean']:6.2f}+/-{r['effective_rank']['std']:.2f}"
            if accs:
                msg += f" probe={r['probe_accuracy']['mean']:.3f}+/-{r['probe_accuracy']['std']:.3f}"
            print(msg, flush=True)

        run("centralized-shared", lambda s: train_model(ds, cen_ep, 1.0, 1.0, seed=s,
                                                        var_on="context", cov_on="context")[0])
        run("federated-shared", lambda s: federated_train(ds, 2, rounds, local_ep, 1.0, 1.0, SHARED, seed=s))
        run("federated-original", lambda s: federated_train(ds, 2, rounds, local_ep, 1.0, 1.0, ORIGINAL, seed=s))

    with open(os.path.join(res_dir, "federated_ms.json"), "w") as f:
        json.dump({"seeds": SEEDS, "results": results}, f, indent=2)
    print("DONE", flush=True)


if __name__ == "__main__":
    main()


## 2. Baselines (structured + digits)


In [ ]:
!python evaluate.py

## 3. CIFAR-10 run


In [ ]:
from evaluate import train_model, collapse_metrics, efficiency_metrics
from dataset import get_dataset
ds=get_dataset("cifar",n_samples=2000,seed=0)
for tag,(a,b) in {"full":(1.0,0.01),"unregularized":(0.0,0.0)}.items():
    m,h,_=train_model(ds,15,a,b,seed=0); cc=collapse_metrics(m,ds)
    print(f"[{tag:13s}] total={h[-1]['total']:.4f} eff_rank={cc['effective_rank']:.2f}")

## 4. Multi-seed robustness (original collapse)


In [ ]:
!python evaluate_multiseed.py

## 5. Shared-embedding fix + linear probe


In [ ]:
!python evaluate_fixes.py

## 6. Regularizer-placement grid (single seed)


In [ ]:
!python evaluate_placement.py

## 6b. Placement grid — 5 seeds (mean ± std)

Reference (digits eff_rank): original 1.02±0.01; shared 34.5±0.2; both-pred 9.2±1.5; swapped 4.4±0.4.


In [ ]:
!python evaluate_placement_ms.py

## 7. Decentralized (2-node FedAvg), single seed


In [ ]:
!python evaluate_federated.py

## 7b. Federated — 3 seeds (mean ± std)

Reference (digits): fed-shared rank 31.0±0.1 acc 0.941±0.006; fed-original rank 1.05±0.03 acc 0.628±0.013.


In [ ]:
!python evaluate_federated_ms.py

## 8. (Optional) CIFAR-10 linear probe across placements


In [ ]:
import numpy as np, torch, torch.nn.functional as F, torchvision
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from dataset import get_dataset
from evaluate import train_model, collapse_metrics
tv=torchvision.datasets.CIFAR10(root="./cifar",train=True,download=True)
imgs=F.interpolate(torch.from_numpy(tv.data[:2000]).float().permute(0,3,1,2)/255.0,size=(64,64),mode="bilinear",align_corners=False)
labels=np.array(tv.targets[:2000]); ds=get_dataset("cifar",n_samples=2000,seed=0)
@torch.no_grad()
def probe(m):
    m.eval(); f=m.context_encoder(imgs).numpy()
    Xtr,Xte,ytr,yte=train_test_split(f,labels,test_size=0.3,random_state=0,stratify=labels)
    s=StandardScaler().fit(Xtr); return LogisticRegression(max_iter=3000).fit(s.transform(Xtr),ytr).score(s.transform(Xte),yte)
for tag,(a,b,von,con) in {"unregularized":(0,0,"context","pred"),"original":(1.0,0.01,"context","pred"),"shared":(1.0,1.0,"context","context")}.items():
    m,h,_=train_model(ds,15,a,b,seed=0,var_on=von,cov_on=con); cc=collapse_metrics(m,ds)
    print(f"[{tag:14s}] eff_rank={cc['effective_rank']:.2f} probe_acc={probe(m):.3f}")